Link to Dataset: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000?resource=download

In [ ]:
!pip install kaggle
!pip install seaborn

Libraries Located Here

In [ ]:
import pandas as pd
import os
from PIL import Image

from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset
from torchvision import transforms
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from collections import Counter


API Key for Dataset

In [ ]:
from google.colab import files

# Directions to get Kaggle API Key!!!:

# 1) Go to kaggle.com and make an account (if you haven't already)
# 2) Go to settings (click profile on top right corner)
# 3) Scroll down to API
# 4) Create new token and name file kaggle.json (should be default name)
# 5) Upload

files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000

!unzip skin-cancer-mnist-ham10000.zip > /dev/null

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.18G/5.20G [00:56<00:00, 191MB/s]
100% 5.20G/5.20G [00:56<00:00, 98.6MB/s]


In [ ]:
!mkdir HAM10000_images
!cp HAM10000_images_part_1/*.jpg HAM10000_images/
!cp HAM10000_images_part_2/*.jpg HAM10000_images/


In [ ]:
df = pd.read_csv('HAM10000_metadata.csv')
df

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear
...,...,...,...,...,...,...,...
10010,HAM_0002867,ISIC_0033084,akiec,histo,40.0,male,abdomen
10011,HAM_0002867,ISIC_0033550,akiec,histo,40.0,male,abdomen
10012,HAM_0002867,ISIC_0033536,akiec,histo,40.0,male,abdomen
10013,HAM_0000239,ISIC_0032854,akiec,histo,80.0,male,face


Feature Engineering, Data Cleaning/Transforming, Splitting Dataset

In [ ]:
# Create missing-age indicator
df['age_missing'] = df['age'].isna().astype(int)

# Fill missing ages with median
median_age = df['age'].median()
df['age'] = df['age'].fillna(median_age)

metadata_path = "HAM10000_metadata.csv"
image_dir = "HAM10000_images"

label_map = {label: idx for idx, label in enumerate(df['dx'].unique())}
df['label'] = df['dx'].map(label_map)

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['label'],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['label'],
    random_state=42
)

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class SkinCancerDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row['image_id'] + ".jpg")
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        metadata = torch.tensor(
            [row['age'], row['age_missing']],
            dtype=torch.float32
        )

        label = torch.tensor(row['label'], dtype=torch.long)

        return image, metadata, label

dataset = {
    "train": SkinCancerDataset(train_df, image_dir, train_transform),
    "val": SkinCancerDataset(val_df, image_dir, eval_transform),
    "test": SkinCancerDataset(test_df, image_dir, eval_transform)
}

In [ ]:
dataset

{'train': <__main__.SkinCancerDataset at 0x7e964e0b31d0>,
 'val': <__main__.SkinCancerDataset at 0x7e964e0b7e30>,
 'test': <__main__.SkinCancerDataset at 0x7e968e348440>}

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(dataset["train"], batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(dataset["val"], batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(dataset["test"], batch_size=BATCH_SIZE, shuffle=False)


CNN Model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

NUM_CLASSES = df['label'].nunique()

class SkinCancerCNN(nn.Module):
    def __init__(self, num_classes):
        super(SkinCancerCNN, self).__init__()

        # ---------- CNN for image ----------
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 112x112

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 56x56

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 28x28
        )

        self.image_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 512),
            nn.ReLU(),
            nn.Dropout(0.5)
        )

        # ---------- Metadata branch ----------
        self.meta_fc = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU()
        )

        # ---------- Final classifier ----------
        self.classifier = nn.Linear(512 + 32, num_classes)

    def forward(self, image, metadata):
        img_features = self.features(image)
        img_features = self.image_fc(img_features)

        meta_features = self.meta_fc(metadata)

        combined = torch.cat((img_features, meta_features), dim=1)
        output = self.classifier(combined)

        return output


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SkinCancerCNN(NUM_CLASSES).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


Epoch Function

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, metadata, labels in loader:
        images = images.to(device)
        metadata = metadata.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images, metadata)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / len(loader), correct / total


def evaluate(model, loader):
    model.eval()
    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, metadata, labels in loader:
            images = images.to(device)
            metadata = metadata.to(device)
            labels = labels.to(device)

            outputs = model(images, metadata)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / len(loader), correct / total


Training Epochs

In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")
    print("-" * 40)


Epoch 1/10
Train Loss: 1.1846, Train Acc: 0.6422
Val Loss:   0.8504, Val Acc:   0.6997
----------------------------------------
Epoch 2/10
Train Loss: 0.8164, Train Acc: 0.7141
Val Loss:   0.7164, Val Acc:   0.7457
----------------------------------------
Epoch 3/10
Train Loss: 0.7374, Train Acc: 0.7354
Val Loss:   0.6876, Val Acc:   0.7463
----------------------------------------


Initial Model Accuracy

In [ ]:
test_loss, test_acc = evaluate(model, test_loader)
print(f"Test Accuracy: {test_acc:.4f}")


Output Results

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# ---------- Collect all predictions and labels ----------
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, metadata, labels in test_loader:
        images = images.to(device)
        metadata = metadata.to(device)
        labels = labels.to(device)

        outputs = model(images, metadata)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# ---------- Confusion Matrix ----------
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_map.keys(), yticklabels=label_map.keys())
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# ---------- Classification Report ----------
print("Classification Report:\n")
print(classification_report(all_labels, all_preds, target_names=label_map.keys()))

# ---------- Per-class Accuracy ----------
per_class_acc = cm.diagonal() / cm.sum(axis=1)
for cls, acc in zip(label_map.keys(), per_class_acc):
    print(f"{cls}: {acc:.4f}")


Exploratory Data Analysis

In [ ]:
# ---------- 1. Label distribution ----------
def plot_label_distribution(split_name, df_split):
    labels = [dataset['train'].df['label'].iloc[i] for i in range(len(df_split))]
    label_counts = Counter(labels)

    plt.figure(figsize=(8,5))
    sns.barplot(x=list(label_counts.keys()), y=list(label_counts.values()))
    plt.title(f"{split_name} Label Distribution")
    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.show()

plot_label_distribution("Train", dataset['train'].df)
plot_label_distribution("Val", dataset['val'].df)
plot_label_distribution("Test", dataset['test'].df)

# ---------- 2. Age distribution ----------
plt.figure(figsize=(8,5))
sns.histplot(dataset['train'].df['age'], bins=20, kde=True)
plt.title("Age Distribution (Train)")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()

# ---------- 3. Sex distribution ----------
plt.figure(figsize=(5,4))
sns.countplot(x='sex', data=dataset['train'].df)
plt.title("Sex Distribution (Train)")
plt.show()

# ---------- 4. Metadata correlation ----------
# Only makes sense for numeric metadata
metadata_cols = ['age', 'age_missing']
plt.figure(figsize=(4,4))
sns.heatmap(dataset['train'].df[metadata_cols].corr(), annot=True, cmap='coolwarm')
plt.title("Metadata Correlation")
plt.show()

# ---------- 5. Example images ----------
def show_sample_images(split_dataset, num=5):
    plt.figure(figsize=(15,3))
    for i in range(num):
        image, metadata, label = split_dataset[i]
        image_np = image.permute(1,2,0).numpy()  # C,H,W -> H,W,C
        plt.subplot(1,num,i+1)
        plt.imshow(np.clip(image_np,0,1))
        plt.title(f"Label: {list(label_map.keys())[label]}")
        plt.axis('off')
    plt.show()

show_sample_images(dataset['train'], num=5)

# ---------- 6. Class balance check ----------
train_labels = [dataset['train'].df['label'].iloc[i] for i in range(len(dataset['train']))]
plt.figure(figsize=(8,4))
sns.histplot(train_labels, bins=len(label_map), discrete=True)
plt.title("Class Balance in Train Split")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

Streamlit Model

In [ ]:
!pip install -q streamlit
!npm install -q -g localtunnel

In [ ]:
# Save the 'state_dict' (the weights) to a file
torch.save(model.state_dict(), 'skin_cancer_model.pth')
print("Model saved as skin_cancer_model.pth")

In [ ]:
%%writefile app.py
import streamlit as st
import torch
import torch.nn as nn
from PIL import Image
import torch.nn.functional as F
from torchvision import transforms
import numpy as np

# 1. DEFINE ARCHITECTURE (Must match your notebook exactly)
class SkinCancerCNN(nn.Module):
    def __init__(self, num_classes):
        super(SkinCancerCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.image_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 512),
            nn.ReLU(),
            nn.Dropout(0.5)
        )
        self.meta_fc = nn.Sequential(nn.Linear(2, 32), nn.ReLU())
        self.classifier = nn.Linear(512 + 32, num_classes)

    def forward(self, image, metadata):
        img_features = self.image_fc(self.features(image))
        meta_features = self.meta_fc(metadata)
        combined = torch.cat((img_features, meta_features), dim=1)
        return self.classifier(combined)

# 2. SETTINGS & LOADING
# List based on your label_map: {label: idx for idx, label in enumerate(df['dx'].unique())}
label_names = ['bkl', 'nv', 'df', 'mel', 'vasc', 'bcc', 'akiec']

@st.cache_resource
def load_trained_model():
    model = SkinCancerCNN(num_classes=7)
    # Load weights saved in Step 1
    model.load_state_dict(torch.load('skin_cancer_model.pth', map_location=torch.device('cpu')))
    model.eval()
    return model

# 3. UI LAYOUT
st.set_page_config(page_title="Skin Cancer AI", layout="centered")
st.title("🩺 Skin Lesion Classifier")
st.markdown("---")

with st.sidebar:
    st.header("Patient Information")
    age = st.slider("Patient Age", 0, 100, 25)
    st.write("This model uses age as a clinical feature to improve prediction accuracy.")

uploaded_file = st.file_uploader("Upload a Dermoscopic Image (JPG/PNG)", type=["jpg", "jpeg", "png"])

if uploaded_file:
    img = Image.open(uploaded_file).convert("RGB")
    st.image(img, caption='Uploaded Image', use_container_width=True)

    if st.button('Run Diagnostic Analysis'):
        model = load_trained_model()

        # Preprocessing (matches your eval_transform)
        transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        image_tensor = transform(img).unsqueeze(0)
        meta_tensor = torch.tensor([[float(age), 0.0]], dtype=torch.float32)

        with st.spinner('Analyzing...'):
            with torch.no_grad():
                output = model(image_tensor, meta_tensor)
                probabilities = F.softmax(output, dim=1)[0]

            # Show top results
            st.subheader("Results")
            top_prob, top_catid = torch.topk(probabilities, 3)

            for i in range(top_prob.size(0)):
                label = label_names[top_catid[i]]
                score = top_prob[i].item()
                st.write(f"**{label.upper()}**")
                st.progress(score)
                st.caption(f"Confidence: {score:.2%}")

!wget -q -O - https://ipv4.icanhazip.com
!streamlit run app.py & npx localtunnel --port 8501